© 2026 Mechanome Bio. Proprietary and confidential.

# Nuclear shape gSEM — step 2 of 3: gene expression vs nuclear shape

**Kernel: R (`mechmap-stats`). Run this notebook in WSL, not native Windows R.**

This applies the MechanoMap gSEM (spatial regression) from `MB_05.2_spatial_regression_v2_extended.ipynb` to **nuclear shape** instead of mechanics. For each of six nuclear shape metrics it asks: *after removing smooth spatial trends and cell size, do cells whose nucleus is (say) more solid express more or less of each gene?*

The statistics are **not re-implemented here**: the notebook sources `mechanomap_nextflow/src/helper_functions.R` and calls its stock `parallel_gsem_regression()` unchanged. All six metrics use the `log` transform, which is exactly what the stock helper applies, so the MB_05.2 per-metric transform fork is not needed.

### How gSEM works, in short
For every gene *g* and metric *m*:
1. Fit a smooth surface over the slide, `s(x, y, k = K_SPATIAL)`, plus a smooth of log cell area, to log(metric) and to normalised expression separately (`mgcv::bam`).
2. Take what is left over (the *residuals*) — the part of each that space and cell size do not explain.
3. Regress the gene residual on the metric residual: `lm(r_gene ~ r_metric)`. The slope is **β**, its p value is corrected with Benjamini–Hochberg **separately for each metric**, and genes with adjusted p < 0.05 are called significant.

### Running in WSL (one-off set-up)
```bash
# in a WSL (Ubuntu) terminal
conda env create -f /mnt/c/MechanomeBioRepos/mechanomap_nextflow/mechmap-stats.yml
conda activate mechmap-stats
Rscript -e "remotes::install_github('caleblareau/BuenColors')"     # only needed by other MB notebooks
Rscript -e "IRkernel::installspec(name='mechmap-stats', displayname='R (mechmap-stats)')"
# every time:
conda activate mechmap-stats
cd /mnt/c/MechanomeBioRepos/nuclear_shape/notebooks
jupyter lab      # then open nuclear_shape_gsem.ipynb and pick the "R (mechmap-stats)" kernel
```
Why WSL: the helper runs genes in parallel with `parallel::makeForkCluster`, which only exists on Linux/macOS. The repositories live on the C: drive, which WSL sees as `/mnt/c/...`, so the relative paths below work unchanged. Run `nuclear_shape_prep.ipynb` (Windows, Python) first.

### Settings carried over from MB_05.2 v2 (Variant E) and choices made for this analysis

| Setting | Value | Note |
|---|---|---|
| Metrics | solidity, extent, shape_index, circularity, perimeter, axis_ratio | measured on Xenium nucleus outlines in the prep notebook |
| Metric transform | `log` for all six | all are strictly positive |
| Winsorising | 0.5 % / 99.5 % tails on each metric | as MB_05.2 `SHAPE_WINSORIZE_FRAC` |
| Normalisation | counts ÷ (Xenium cell area ÷ median cell area), then log1p | as MB_05.2 |
| Cell area floor | cells below the 1st percentile of cell area dropped | as MB_05.2 (`MIN_CELL_AREA = NULL`) |
| Size covariate | `s(log_cell_area, k = 10)` in every smooth | **cell** area, not nuclear area: perimeter β therefore still contains nuclear-size effects |
| Spatial smooth | `s(centroid_x, centroid_y, k = 300)`, `bam(fREML, discrete = TRUE)`, not fixed df | as MB_05.2 |
| Gene filter | detected in ≥ 5 % of cells | all such genes are tested |
| Multiple testing | Benjamini–Hochberg per metric; significant = padj < 0.05 | |
| Cell-type post-hoc | pancreas only, `cell_type` column; types with < 50 cells skipped | uses saved residuals; not run for lung (no labels) |


### Settings

In [1]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# ===== Choose the tissue =====
# Run the whole notebook once with "pancreas" and once with "lung".
DATASET <- "lung"

# ===== Folders (relative to nuclear_shape/notebooks, the folder this notebook is in) =====
REPOSITORY_FOLDER     <- normalizePath("..")
HELPER_FUNCTIONS_PATH <- file.path(REPOSITORY_FOLDER, "..", "mechanomap_nextflow", "src", "helper_functions.R")
PREP_FOLDER      <- file.path(REPOSITORY_FOLDER, "outputs", DATASET, "prep")       # written by the prep notebook
GSEM_FOLDER      <- file.path(REPOSITORY_FOLDER, "outputs", DATASET, "gsem")       # results tables + residuals
PLOT_DATA_FOLDER <- file.path(REPOSITORY_FOLDER, "outputs", DATASET, "plot_data")  # everything the plots notebook needs

PROPRIETARY_NOTICE <- "\u00a9 2026 Mechanome Bio. Proprietary and confidential."  # \u00a9 is the © sign (written this way so it survives any locale)

# ===== Shape metrics =====
SHAPE_METRICS <- c("solidity", "extent", "shape_index", "circularity", "perimeter", "axis_ratio")

# ===== Normalisation and filtering (as MB_05.2) =====
MIN_CELL_AREA  <- NULL     # NULL = drop cells below the 1st percentile of cell area; a number = hard floor in µm²
WINSORIZE_FRAC <- 0.005    # clip each shape metric at the 0.5% and 99.5% quantiles; NULL = no clipping
EXPR_FRAC_MIN  <- 0.05     # a gene must be detected (count > 0) in at least this fraction of cells
N_GENES_SAMPLE <- NULL     # NULL = all expressed genes; a number = random subset for a quick trial run
RANDOM_SEED    <- 42

# ===== gSEM model (MB_05.2 Variant E) =====
K_SPATIAL         <- 300       # basis size of the 2D spatial smooth
MODEL_FX          <- FALSE     # FALSE = penalised smooth (wiggliness chosen from the data)
USE_BAM           <- TRUE      # bam(discrete = TRUE) for speed
gSEM_METHOD       <- "fREML"
INCLUDE_AREA_COV  <- TRUE      # adds s(log_cell_area) to every smooth
K_AREA_COV        <- 10
NCORES            <- 6L        # parallel workers; each needs roughly 0.4-0.6 GB of memory
CHECKPOINT_EVERY  <- 100L      # write results to disk every N genes (lets a crashed run resume)

# TRUE = delete old residual batches and the checkpoint, so the run starts clean.
# Leave TRUE unless you are deliberately resuming a crashed run of the SAME cells. (The stock
# helper numbers its residual batch files from 1 again on resume, so a resumed run overwrites
# the residuals of earlier genes; the main results are unaffected, the cell-type step loses them.)
FORCE_RERUN_RESIDUALS <- TRUE

# ===== Multiple testing =====
PADJ_METHOD     <- "BH"
PADJ_PER_METRIC <- TRUE        # a separate Benjamini-Hochberg correction for each metric
SIGNIFICANCE_PADJ <- 0.05

# ===== Plot data =====
N_LABEL_PER_SIDE <- 15         # genes marked for labelling on each side of each volcano plot

# ===== Cell-type post-hoc (only runs if the prep table has a cell_type column) =====
RUN_CELL_TYPE_ANALYSIS <- TRUE
MIN_CELLS_PER_TYPE     <- 50   # cell types with fewer labelled cells are skipped in the stratified analysis

CHECKPOINT_FILE <- file.path(GSEM_FOLDER, paste0(DATASET, "_gsem_checkpoint.csv"))
RESIDUALS_DIR   <- file.path(GSEM_FOLDER, "gsem_residuals")
for (folder in c(GSEM_FOLDER, PLOT_DATA_FOLDER)) dir.create(folder, recursive = TRUE, showWarnings = FALSE)

cat(sprintf("Dataset: %s | metrics: %s\nNCORES: %d | K_SPATIAL: %d\n",
            DATASET, paste(SHAPE_METRICS, collapse = ", "), NCORES, K_SPATIAL))

Dataset: lung | metrics: solidity, extent, shape_index, circularity, perimeter, axis_ratio
NCORES: 6 | K_SPATIAL: 300


### Libraries, helper functions and output writers

In [2]:
# © 2026 Mechanome Bio. Proprietary and confidential.
# Stop early on native Windows: the stock helper needs fork-based parallelism (Linux / WSL / macOS).
if (.Platform$OS.type == "windows") {
  stop("This notebook must run in WSL (or Linux/macOS): helper_functions.R uses ",
       "parallel::makeForkCluster, which is not available in native Windows R. ",
       "See the set-up instructions at the top of the notebook.")
}

suppressWarnings(suppressMessages({
  library(mgcv)
  library(Matrix)
  library(data.table)
  library(dplyr)
  library(parallel)
  library(doParallel)   # also attaches foreach and iterators (icount), which the helper uses
}))

if (!file.exists(HELPER_FUNCTIONS_PATH))
  stop("helper_functions.R not found at ", HELPER_FUNCTIONS_PATH)
source(HELPER_FUNCTIONS_PATH)   # defines .fit_spatial(), parallel_gsem_regression(), do_gsem_regression()
cat("Sourced", normalizePath(HELPER_FUNCTIONS_PATH), "\n")

# ---- small writers that put the proprietary notice in every output file ----

# CSV whose first line is "# © 2026 ..."; Python reads it back with pandas.read_csv(path, comment="#").
write_csv_with_notice <- function(data, path, row.names = FALSE) {
  connection <- file(path, open = "wb")   # binary mode: bytes are written exactly as given (UTF-8 notice, \n line endings)
  on.exit(close(connection))
  writeLines(enc2utf8(paste("#", PROPRIETARY_NOTICE)), connection, useBytes = TRUE)
  write.csv(data, connection, row.names = row.names)
}

# Minimal JSON writer (jsonlite is not in the mechmap-stats environment).
# Handles NULL, numbers, logicals, text, vectors, and (named) lists.
to_json <- function(value, indent = 0) {
  pad  <- strrep("  ", indent)
  pad2 <- strrep("  ", indent + 1)
  quote_text <- function(text) {
    text <- gsub("\\\\", "\\\\\\\\", text)
    text <- gsub("\"", "\\\\\"", text)
    text <- gsub("\n", "\\\\n", text)
    paste0("\"", text, "\"")
  }
  scalar <- function(single) {
    if (is.na(single)) return("null")
    if (is.logical(single)) return(if (single) "true" else "false")
    if (is.numeric(single)) return(format(single, digits = 15))
    quote_text(as.character(single))
  }
  if (is.null(value)) return("null")
  if (is.list(value)) {
    if (length(value) == 0) return(if (is.null(names(value))) "[]" else "{}")
    inner <- vapply(seq_along(value), function(i) to_json(value[[i]], indent + 1), character(1))
    if (!is.null(names(value)))
      return(paste0("{\n", paste0(pad2, quote_text(names(value)), ": ", inner, collapse = ",\n"), "\n", pad, "}"))
    return(paste0("[\n", paste0(pad2, inner, collapse = ",\n"), "\n", pad, "]"))
  }
  if (length(value) == 1) return(scalar(value))
  paste0("[", paste(vapply(as.list(value), scalar, character(1)), collapse = ", "), "]")
}

write_json_with_notice <- function(named_list, path) {
  named_list <- c(list(proprietary_notice = PROPRIETARY_NOTICE), named_list)
  writeLines(enc2utf8(to_json(named_list)), path, useBytes = TRUE)
}

# Reads a one-value-per-line text file, skipping the "# notice" line.
read_list_skipping_notice <- function(path) {
  lines <- readLines(path, encoding = "UTF-8")
  lines[!startsWith(lines, "#") & nzchar(lines)]
}
cat("Libraries and writers ready\n")

Sourced /mnt/c/MechanomeBioRepos/mechanomap_nextflow/src/helper_functions.R 
Libraries and writers ready


### Load the prepared data

Reads what `nuclear_shape_prep.ipynb` wrote: one row per cell (position, cell area, shape metrics, optional cell type) and the raw cells × genes count matrix in the same row order.

In [3]:
# © 2026 Mechanome Bio. Proprietary and confidential.
cell_data <- read.csv(file.path(PREP_FOLDER, "cells.csv"), skip = 1, check.names = FALSE,
                      stringsAsFactors = FALSE)
count_cells <- read_list_skipping_notice(file.path(PREP_FOLDER, "counts_cells.txt"))
count_genes <- read_list_skipping_notice(file.path(PREP_FOLDER, "counts_genes.txt"))

gex_sparse <- as(Matrix::readMM(file.path(PREP_FOLDER, "counts.mtx")), "CsparseMatrix")
dimnames(gex_sparse) <- list(count_cells, count_genes)

if (!identical(cell_data$cell_id, count_cells))
  stop("cells.csv and counts_cells.txt are not in the same order - re-run the prep notebook")
missing_metrics <- setdiff(SHAPE_METRICS, names(cell_data))
if (length(missing_metrics) > 0)
  stop("cells.csv is missing metric column(s): ", paste(missing_metrics, collapse = ", "))

# The helper looks up cells by row name, so the cell IDs become the row names.
rownames(cell_data) <- cell_data$cell_id
HAS_CELL_TYPES <- "cell_type" %in% names(cell_data)

cat(sprintf("Loaded %s cells x %d genes for %s (cell types: %s)\n",
            format(nrow(gex_sparse), big.mark = ","), ncol(gex_sparse), DATASET,
            if (HAS_CELL_TYPES) "yes" else "no"))
head(cell_data[, c("cell_id", "centroid_x", "centroid_y", "cell_area", SHAPE_METRICS)])

Loaded 140,259 cells x 377 genes for lung (cell types: no)


,cell_id,centroid_x,centroid_y,cell_area,solidity,extent,shape_index,circularity,perimeter,axis_ratio
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
fdjjgoig-1,fdjjgoig-1,1545.592,2414.922,496.7188,0.9911472,0.7957756,3.636938,0.9500315,58.40270,0.8382122
hgafogjf-1,hgafogjf-1,2399.042,1804.835,343.2778,0.9530548,0.6876922,3.896044,0.8278701,59.04043,0.9378864
hdjkncho-1,hdjkncho-1,2495.690,2106.159,327.9247,0.9636682,0.6702167,3.901311,0.8256360,58.85515,0.8801945
fehenekb-1,fehenekb-1,1328.358,2512.546,323.9961,0.9488443,0.7686496,3.997481,0.7863882,57.90382,0.6372341
lkagnbkp-1,lkagnbkp-1,3905.267,2378.672,278.0270,0.9084137,0.6715186,5.258677,0.4544193,75.11175,0.4471887
hhcpfkdh-1,hhcpfkdh-1,2266.316,1650.837,290.9417,0.9762288,0.7385024,3.811507,0.8650004,53.94206,0.8507910


### Cell-area floor and normalisation (as MB_05.2)

Counts are divided by each cell's Xenium area relative to the median area, then `log1p` is applied. The same cell area is used again as the `s(log_cell_area)` covariate inside gSEM.

In [4]:
# © 2026 Mechanome Bio. Proprietary and confidential.
area_floor <- if (is.null(MIN_CELL_AREA)) quantile(cell_data$cell_area, 0.01, names = FALSE) else MIN_CELL_AREA
keep_cell  <- is.finite(cell_data$cell_area) & cell_data$cell_area >= area_floor
cat(sprintf("Dropping %d cells with cell area < %.1f µm²\n", sum(!keep_cell), area_floor))
cell_data  <- cell_data[keep_cell, ]
gex_sparse <- gex_sparse[keep_cell, , drop = FALSE]

size_factors <- cell_data$cell_area / median(cell_data$cell_area)
saved_cell_ids <- rownames(gex_sparse)                           # Diagonal %*% drops row names
gex_sparse <- Matrix::Diagonal(x = 1 / size_factors) %*% gex_sparse
rownames(gex_sparse) <- saved_cell_ids
gex_sparse@x <- log1p(gex_sparse@x)                              # log1p(0) = 0, so zeros stay zero
cell_data$log_cell_area <- log(cell_data$cell_area)

cat(sprintf("Normalised by Xenium cell area (median %.1f µm²): %s cells x %d genes\n",
            median(cell_data$cell_area), format(nrow(gex_sparse), big.mark = ","), ncol(gex_sparse)))

Dropping 1401 cells with cell area < 15.6 µm²
Normalised by Xenium cell area (median 62.5 µm²): 138,858 cells x 377 genes


### Winsorise the shape metrics

Clips each metric at its 0.5 % and 99.5 % quantiles so a few odd outlines cannot dominate the log-scale smooth. Values are then floored at 1e-12 so `log()` is always finite (all six metrics are already positive, so this floor should never bind).

In [5]:
# © 2026 Mechanome Bio. Proprietary and confidential.
winsorize <- function(values, fraction) {
  limits <- quantile(values, c(fraction, 1 - fraction), na.rm = TRUE)
  pmin(pmax(values, limits[1]), limits[2])
}

for (metric in SHAPE_METRICS) {
  before <- cell_data[[metric]]
  if (any(!is.finite(before) | before <= 0))
    stop(sprintf("metric '%s' has %d non-finite or non-positive value(s); log() would fail",
                 metric, sum(!is.finite(before) | before <= 0)))
  after <- if (is.null(WINSORIZE_FRAC)) before else winsorize(before, WINSORIZE_FRAC)
  cell_data[[metric]] <- pmax(after, 1e-12)
  cat(sprintf("  %-12s [%.4g, %.4g] -> [%.4g, %.4g]\n", metric,
              min(before), max(before), min(cell_data[[metric]]), max(cell_data[[metric]])))
}

  solidity     [0.4198, 1] -> [0.6977, 0.9955]
  extent       [0.07807, 0.9375] -> [0.3155, 0.8406]
  shape_index  [3.568, 10.54] -> [3.629, 6.129]
  circularity  [0.1132, 0.987] -> [0.3346, 0.954]
  perimeter    [8.157, 89.91] -> [9.432, 46]
  axis_ratio   [0.06032, 0.9988] -> [0.1894, 0.9658]


### Metric QC data (saved for plotting)

- `metric_values.csv` — each cell's six metrics exactly as they enter the model (after winsorising, before `log`), plus log cell area.
- `metric_collinearity_before_detrending.csv` — Spearman correlation between log metrics and log cell area, before the spatial smooth. The correlation **after** the smooth (the one that matters for interpreting overlapping hits) is saved once gSEM has run.

In [6]:
# © 2026 Mechanome Bio. Proprietary and confidential.
write_csv_with_notice(cell_data[, c("cell_id", SHAPE_METRICS, "log_cell_area")],
                      file.path(PLOT_DATA_FOLDER, "metric_values.csv"))

log_metric_matrix <- sapply(SHAPE_METRICS, function(metric) log(cell_data[[metric]]))
log_metric_matrix <- cbind(log_metric_matrix, log_cell_area = cell_data$log_cell_area)
collinearity_before <- cor(log_metric_matrix, method = "spearman", use = "complete.obs")
write_csv_with_notice(as.data.frame(collinearity_before),
                      file.path(PLOT_DATA_FOLDER, "metric_collinearity_before_detrending.csv"),
                      row.names = TRUE)
print(round(collinearity_before, 2))
rm(log_metric_matrix)

              solidity extent shape_index circularity perimeter axis_ratio
solidity          1.00   0.65       -0.84        0.84      0.03       0.43
extent            0.65   1.00       -0.75        0.75     -0.21       0.60
shape_index      -0.84  -0.75        1.00       -1.00      0.11      -0.77
circularity       0.84   0.75       -1.00        1.00     -0.11       0.77
perimeter         0.03  -0.21        0.11       -0.11      1.00      -0.24
axis_ratio        0.43   0.60       -0.77        0.77     -0.24       1.00
log_cell_area     0.21  -0.02       -0.08        0.08      0.78      -0.09
              log_cell_area
solidity               0.21
extent                -0.02
shape_index           -0.08
circularity            0.08
perimeter              0.78
axis_ratio            -0.09
log_cell_area          1.00


### Choose the genes

In [7]:
# © 2026 Mechanome Bio. Proprietary and confidential.
expressed_fraction <- Matrix::colSums(gex_sparse > 0) / nrow(gex_sparse)
expressed_genes    <- colnames(gex_sparse)[expressed_fraction >= EXPR_FRAC_MIN]

set.seed(RANDOM_SEED)
genes_use <- if (!is.null(N_GENES_SAMPLE) && N_GENES_SAMPLE < length(expressed_genes)) {
  sample(expressed_genes, N_GENES_SAMPLE)
} else {
  expressed_genes
}
cat(sprintf("Expressed genes (detected in >= %.0f%% of cells): %d of %d | tested: %d\n",
            100 * EXPR_FRAC_MIN, length(expressed_genes), ncol(gex_sparse), length(genes_use)))

Expressed genes (detected in >= 5% of cells): 169 of 377 | tested: 169


### Run gSEM

Calls the stock `parallel_gsem_regression()` from `helper_functions.R`. It fits the six metric smooths once, then for each gene fits one expression smooth and six `lm()` calls, checkpointing to disk every `CHECKPOINT_EVERY` genes. **This is the slow step** (on the order of an hour per tissue, depending on cell count and cores). Its outputs are saved, so the cells below can be re-run without repeating it — see the reload cell.

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
if (FORCE_RERUN_RESIDUALS) {
  old_batches <- list.files(RESIDUALS_DIR, pattern = "^r_gene_batch_", full.names = TRUE)
  if (length(old_batches) > 0) file.remove(old_batches)
  if (file.exists(CHECKPOINT_FILE)) file.remove(CHECKPOINT_FILE)
  cat(sprintf("Clean run: removed %d old residual batch file(s) and any old checkpoint\n", length(old_batches)))
}

# The helper wants genes as ROWS and cells as COLUMNS (a dense table).
gex_res_dense <- as.data.frame(t(as.matrix(gex_sparse[, genes_use, drop = FALSE])))
cat(sprintf("Dense expression table: %d genes x %d cells (%s)\n",
            nrow(gex_res_dense), ncol(gex_res_dense), format(object.size(gex_res_dense), units = "Gb")))

gsem_start <- Sys.time()
gsem_results <- parallel_gsem_regression(
  genes_use, cell_data, gex_res_dense,
  k_sp             = K_SPATIAL,
  model_fx         = MODEL_FX,
  method           = gSEM_METHOD,
  ncores           = NCORES,
  use_bam          = USE_BAM,
  metrics          = SHAPE_METRICS,       # the helper applies log() to each metric
  include_area_cov = INCLUDE_AREA_COV,
  k_area           = K_AREA_COV,
  checkpoint_file  = CHECKPOINT_FILE,
  checkpoint_every = CHECKPOINT_EVERY,
  residuals_dir    = RESIDUALS_DIR)
gsem_minutes <- as.numeric(difftime(Sys.time(), gsem_start, units = "mins"))
cat(sprintf("gSEM complete in %.1f minutes: %d rows\n", gsem_minutes, nrow(gsem_results)))

Clean run: removed 0 old residual batch file(s) and any old checkpoint
Dense expression table: 169 genes x 138858 cells (0.2 Gb)


### Multiple-testing correction and results tables

If you restarted the kernel after gSEM finished, run the settings, libraries, load and normalisation cells, then this cell: it reloads the results from the checkpoint file instead of re-running gSEM.

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
if (!exists("gsem_results")) {
  cat("gsem_results not in memory - reloading from the checkpoint file\n")
  gsem_results <- as.data.frame(data.table::fread(CHECKPOINT_FILE))
  if (!exists("gsem_minutes")) gsem_minutes <- NA_real_
}
gsem_results <- as.data.frame(gsem_results)
gsem_results <- gsem_results[!duplicated(gsem_results[, c("gene", "metric")]), ]

gsem_results$padj <- NA_real_
if (PADJ_PER_METRIC) {
  for (metric in unique(gsem_results$metric)) {
    rows <- gsem_results$metric == metric
    gsem_results$padj[rows] <- p.adjust(gsem_results$pval[rows], method = PADJ_METHOD)
  }
} else {
  gsem_results$padj <- p.adjust(gsem_results$pval, method = PADJ_METHOD)
}
gsem_results$significant <- !is.na(gsem_results$padj) & gsem_results$padj < SIGNIFICANCE_PADJ

write_csv_with_notice(gsem_results, file.path(GSEM_FOLDER, paste0(DATASET, "_nuclear_shape_gsem_results.csv")))

# ---- hit counts per metric (plot data) ----
hit_counts <- do.call(rbind, lapply(SHAPE_METRICS, function(metric) {
  rows <- gsem_results[gsem_results$metric == metric, ]
  hits <- rows[rows$significant, ]
  data.frame(metric = metric,
             genes_tested      = nrow(rows),
             genes_significant = nrow(hits),
             genes_positive_beta = sum(hits$beta > 0, na.rm = TRUE),
             genes_negative_beta = sum(hits$beta < 0, na.rm = TRUE),
             genes_failed      = sum(!is.na(rows$error_msg)),
             smallest_padj     = suppressWarnings(min(rows$padj, na.rm = TRUE)),
             stringsAsFactors = FALSE)
}))
write_csv_with_notice(hit_counts, file.path(PLOT_DATA_FOLDER, "hit_counts.csv"))
print(hit_counts)

# ---- overlap: how many significant genes each pair of metrics shares (plot data) ----
significant_genes <- lapply(setNames(SHAPE_METRICS, SHAPE_METRICS), function(metric)
  unique(gsem_results$gene[gsem_results$metric == metric & gsem_results$significant]))
hit_overlap <- outer(SHAPE_METRICS, SHAPE_METRICS, Vectorize(function(first, second)
  length(intersect(significant_genes[[first]], significant_genes[[second]]))))
dimnames(hit_overlap) <- list(SHAPE_METRICS, SHAPE_METRICS)
write_csv_with_notice(as.data.frame(hit_overlap), file.path(PLOT_DATA_FOLDER, "hit_overlap.csv"), row.names = TRUE)
print(hit_overlap)
cat(sprintf("\nGenes significant for at least one metric: %d of %d tested\n",
            length(unique(unlist(significant_genes))), length(unique(gsem_results$gene))))

### Volcano plot data

One table per metric: every gene's β, raw and adjusted p value, whether it is significant, and whether it is one of the top `N_LABEL_PER_SIDE` significant genes on each side (by adjusted p) to be labelled.

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
make_volcano_table <- function(results, metric) {
  table <- results[results$metric == metric & !is.na(results$padj),
                   c("gene", "beta", "stat", "pval", "padj", "significant")]
  # padj can underflow to exactly 0 for extremely strong genes; floor it so -log10 stays finite.
  table$neg_log10_padj <- -log10(pmax(table$padj, .Machine$double.xmin))
  hits <- table[table$significant, ]
  top_positive <- head(hits$gene[hits$beta > 0][order(hits$padj[hits$beta > 0])], N_LABEL_PER_SIDE)
  top_negative <- head(hits$gene[hits$beta < 0][order(hits$padj[hits$beta < 0])], N_LABEL_PER_SIDE)
  table$label <- table$gene %in% c(top_positive, top_negative)
  table[order(table$padj), ]
}

for (metric in SHAPE_METRICS) {
  volcano_table <- make_volcano_table(gsem_results, metric)
  write_csv_with_notice(volcano_table, file.path(PLOT_DATA_FOLDER, paste0("volcano_", metric, ".csv")))
  cat(sprintf("  %-12s %4d genes, %4d significant\n", metric, nrow(volcano_table), sum(volcano_table$significant)))
}

### Correlation between the metrics after spatial detrending

Shape metrics are geometrically related (for example `circularity` = 4π / `shape_index`², so after `log` they are perfectly negatively correlated and will give mirror-image hits). This is the correlation between the metric residuals that gSEM actually regresses on.

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
metric_residuals <- readRDS(file.path(RESIDUALS_DIR, "r_metrics.rds"))
collinearity_after <- cor(do.call(cbind, metric_residuals), method = "spearman", use = "complete.obs")
write_csv_with_notice(as.data.frame(collinearity_after),
                      file.path(PLOT_DATA_FOLDER, "metric_collinearity_after_detrending.csv"),
                      row.names = TRUE)
print(round(collinearity_after, 2))

### Cell-type post-hoc analysis (pancreas only)

Different cell types have different nuclear shapes *and* different marker genes, so a slide-wide hit can simply mean "this gene marks a cell type whose nuclei look like that". Using the residuals gSEM saved (no new spatial fits):

- **Stratified**: `lm(r_gene ~ r_metric)` separately within each cell type (types with fewer than `MIN_CELLS_PER_TYPE` labelled cells skipped). BH correction within each cell type × metric.
- **Blocked**: `lm(r_gene ~ r_metric + cell_type)` on all labelled cells — the slide-wide β after allowing each cell type its own baseline expression. BH correction per metric.

Only cells that passed MechanoMap carry a label in the h5ad; unlabelled cells are left out of this step only. Same approach as MB_05.2.

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
if (RUN_CELL_TYPE_ANALYSIS && HAS_CELL_TYPES) {

  # ---- reload residuals; columns of each batch are gene names ----
  cell_order    <- readLines(file.path(RESIDUALS_DIR, "cell_order.txt"))
  batch_files   <- sort(list.files(RESIDUALS_DIR, pattern = "^r_gene_batch_.*\\.rds$", full.names = TRUE))
  gene_residuals <- do.call(cbind, lapply(batch_files, readRDS))
  rownames(gene_residuals) <- cell_order
  gene_residuals <- gene_residuals[, !duplicated(colnames(gene_residuals)), drop = FALSE]
  if (ncol(gene_residuals) < length(unique(gsem_results$gene)))
    warning(sprintf("Residuals found for %d of %d genes (a resumed run overwrites batch files); ",
                    ncol(gene_residuals), length(unique(gsem_results$gene))),
            "the cell-type step uses only those genes.")

  cell_types <- cell_data[cell_order, "cell_type"]
  cell_types[cell_types %in% c("", "nan", "NA")] <- NA
  labelled   <- !is.na(cell_types)
  type_sizes <- table(cell_types[labelled])
  cat(sprintf("Labelled cells: %d of %d\n", sum(labelled), length(labelled)))
  print(type_sizes)
  types_to_test <- names(type_sizes)[type_sizes >= MIN_CELLS_PER_TYPE]
  cat(sprintf("Cell types with >= %d cells (tested): %s\n", MIN_CELLS_PER_TYPE, paste(types_to_test, collapse = ", ")))

  slope_and_p <- function(model) {
    coefficients <- summary(model)$coefficients
    if (!"metric_residual" %in% rownames(coefficients)) return(c(NA_real_, NA_real_))
    coefficients["metric_residual", c(1, 4)]
  }

  # ---- stratified: one lm per cell type x gene x metric ----
  stratified_results <- do.call(rbind, lapply(types_to_test, function(cell_type) {
    rows <- which(cell_types == cell_type)
    do.call(rbind, lapply(colnames(gene_residuals), function(gene) {
      gene_residual <- gene_residuals[rows, gene]
      do.call(rbind, lapply(names(metric_residuals), function(metric) {
        metric_residual <- metric_residuals[[metric]][rows]
        estimate <- slope_and_p(lm(gene_residual ~ metric_residual))
        data.frame(cell_type = cell_type, gene = gene, metric = metric, n_cells = length(rows),
                   beta = estimate[1], pval = estimate[2], stringsAsFactors = FALSE)
      }))
    }))
  }))
  stratified_results$padj <- NA_real_
  for (cell_type in types_to_test) for (metric in names(metric_residuals)) {
    rows <- stratified_results$cell_type == cell_type & stratified_results$metric == metric
    stratified_results$padj[rows] <- p.adjust(stratified_results$pval[rows], method = PADJ_METHOD)
  }
  stratified_results$significant <- !is.na(stratified_results$padj) & stratified_results$padj < SIGNIFICANCE_PADJ
  write_csv_with_notice(stratified_results, file.path(GSEM_FOLDER, paste0(DATASET, "_gsem_stratified_by_cell_type.csv")))

  # ---- blocked: cell type as a covariate on all labelled cells ----
  cell_type_factor <- factor(cell_types[labelled])
  blocked_results <- do.call(rbind, lapply(colnames(gene_residuals), function(gene) {
    gene_residual <- gene_residuals[labelled, gene]
    do.call(rbind, lapply(names(metric_residuals), function(metric) {
      metric_residual <- metric_residuals[[metric]][labelled]
      estimate <- slope_and_p(lm(gene_residual ~ metric_residual + cell_type_factor))
      data.frame(gene = gene, metric = metric, beta = estimate[1], pval = estimate[2],
                 error_msg = NA_character_, stringsAsFactors = FALSE)
    }))
  }))
  blocked_results$padj <- NA_real_
  for (metric in names(metric_residuals)) {
    rows <- blocked_results$metric == metric
    blocked_results$padj[rows] <- p.adjust(blocked_results$pval[rows], method = PADJ_METHOD)
  }
  blocked_results$significant <- !is.na(blocked_results$padj) & blocked_results$padj < SIGNIFICANCE_PADJ
  blocked_results$stat <- NA_real_
  write_csv_with_notice(blocked_results, file.path(GSEM_FOLDER, paste0(DATASET, "_gsem_blocked_by_cell_type.csv")))

  # ---- plot data: hits per cell type, blocked volcanoes, slide-wide vs blocked betas ----
  cell_type_hit_counts <- aggregate(significant ~ cell_type + metric, data = stratified_results, FUN = sum)
  names(cell_type_hit_counts)[names(cell_type_hit_counts) == "significant"] <- "genes_significant"
  cell_type_hit_counts$n_cells <- as.integer(type_sizes[cell_type_hit_counts$cell_type])
  write_csv_with_notice(cell_type_hit_counts, file.path(PLOT_DATA_FOLDER, "cell_type_hit_counts.csv"))

  for (metric in SHAPE_METRICS)
    write_csv_with_notice(make_volcano_table(blocked_results, metric),
                          file.path(PLOT_DATA_FOLDER, paste0("volcano_blocked_", metric, ".csv")))

  beta_comparison <- merge(gsem_results[, c("gene", "metric", "beta", "padj")],
                           blocked_results[, c("gene", "metric", "beta", "padj")],
                           by = c("gene", "metric"), suffixes = c("_slide_wide", "_cell_type_blocked"))
  write_csv_with_notice(beta_comparison, file.path(PLOT_DATA_FOLDER, "beta_slide_wide_vs_cell_type_blocked.csv"))

  cat(sprintf("\nStratified: %d rows | blocked: %d rows | significant (blocked) per metric:\n",
              nrow(stratified_results), nrow(blocked_results)))
  print(tapply(blocked_results$significant, blocked_results$metric, sum))
  rm(gene_residuals); gc()
} else {
  cat("Cell-type post-hoc skipped (", if (!HAS_CELL_TYPES) "no cell_type column for this tissue" else "RUN_CELL_TYPE_ANALYSIS = FALSE", ")\n")
}

### Run record

Every setting, cell and gene count, and package version for this run, with the proprietary notice, so the figures can be traced back to exactly how they were made.

In [ ]:
# © 2026 Mechanome Bio. Proprietary and confidential.
run_record <- list(
  created  = format(Sys.time(), "%Y-%m-%dT%H:%M:%S"),
  notebook = "notebooks/nuclear_shape_gsem.ipynb",
  dataset  = DATASET,
  helper_functions = normalizePath(HELPER_FUNCTIONS_PATH),
  gsem_driver = "helper_functions.R::parallel_gsem_regression (stock, unmodified)",
  shape_metrics = SHAPE_METRICS,
  metric_transform = "log (all metrics)",
  winsorize_fraction = WINSORIZE_FRAC,
  normalisation = "counts / (Xenium cell_area / median cell_area), then log1p",
  cell_area_floor_square_microns = unname(area_floor),
  expression_fraction_minimum = EXPR_FRAC_MIN,
  n_genes_sample = N_GENES_SAMPLE,
  k_spatial = K_SPATIAL, model_fx = MODEL_FX, use_bam = USE_BAM, method = gSEM_METHOD,
  include_area_covariate = INCLUDE_AREA_COV, area_covariate = "log_cell_area", k_area = K_AREA_COV,
  ncores = NCORES,
  padj_method = PADJ_METHOD, padj_per_metric = PADJ_PER_METRIC, significance_padj = SIGNIFICANCE_PADJ,
  test = "lm(r_gene ~ r_metric) on spatial + cell-area smooth residuals (gSEM)",
  cell_type_analysis = RUN_CELL_TYPE_ANALYSIS && HAS_CELL_TYPES,
  min_cells_per_type = MIN_CELLS_PER_TYPE,
  n_cells = nrow(cell_data),
  n_genes_tested = length(genes_use),
  gsem_minutes = gsem_minutes,
  r_version = R.version.string,
  package_versions = list(mgcv = as.character(packageVersion("mgcv")),
                          Matrix = as.character(packageVersion("Matrix")),
                          data.table = as.character(packageVersion("data.table")))
)
write_json_with_notice(run_record, file.path(GSEM_FOLDER, "run_metadata.json"))
write_json_with_notice(run_record, file.path(PLOT_DATA_FOLDER, "plot_data_metadata.json"))
cat("Run record written to", GSEM_FOLDER, "and", PLOT_DATA_FOLDER, "\n")
sessionInfo()

### Next step

Run `nuclear_shape_gsem.ipynb` again with `DATASET <- "lung"`, then open `nuclear_shape_plots.ipynb` (Windows, Python) to draw the figures from `outputs/<tissue>/plot_data/`.

© 2026 Mechanome Bio. Proprietary and confidential.